# PUL7SAR Phase 18 — Golden Visual GPU Proof
This notebook runs the existing Phase 18 `$0-local` pipeline on the notebook GPU. It does not use a paid image API. Select a GPU runtime before running. The first run generates candidate 1 only; generate the full four-seed batch only after candidate 1 proves the runtime is stable.

In [ ]:
import os, subprocess, sys
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
assert subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0, 'GPU runtime is required'


In [ ]:
!git clone --depth 1 --branch phase18/story-intelligence https://github.com/pulsar7official/pul7sar-bot.git
%cd pul7sar-bot
!python -m pip install -U diffusers transformers accelerate safetensors Pillow


In [ ]:
!PYTHONPATH=. python tools/phase18_local_readiness.py


In [ ]:
!PYTHONPATH=. python tools/phase18_build_golden_batch.py --output-dir output/phase18_handoffs/golden-batch --seeds 7007001 7007002 7007003 7007004
!cat output/phase18_handoffs/golden-batch/manifest.json


## Generate candidate 1
This may download the open model weights on the first run. The executor re-checks CUDA, VRAM, `Flux2KleinPipeline`, handoff SHA-256, provider/model identity, seed, and canvas contract before generation.

In [ ]:
!PYTHONPATH=. python tools/phase18_flux2_execute.py --request output/phase18_handoffs/golden-batch/candidate-01-seed-7007001.json --generation-dir output/phase18_generated --proof-dir output/phase18_visual_proof --dtype bfloat16


In [ ]:
from pathlib import Path
from IPython.display import display, Image
proofs = sorted(Path('output/phase18_visual_proof').glob('*.png'))
assert proofs, 'No real visual proof PNG was generated'
print(proofs[-1])
display(Image(filename=str(proofs[-1])))


## Optional: generate all four deterministic candidates
Run this only after candidate 1 succeeds technically. Candidates execute sequentially to avoid VRAM contention.

In [ ]:
# Uncomment after the first candidate succeeds:
# !PYTHONPATH=. python tools/phase18_flux2_batch_execute.py --manifest output/phase18_handoffs/golden-batch/manifest.json --generation-dir output/phase18_generated --proof-dir output/phase18_visual_proof --dtype bfloat16 --result output/phase18_visual_proof/batch-execution.json
